In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import (
    KBinsDiscretizer,
    OneHotEncoder,
    PowerTransformer,
)
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.pipeline import Pipeline
import json

In [ ]:
X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [ ]:
ct = ColumnTransformer(
    [
        (
            "Categorical columns",
            OneHotEncoder(handle_unknown="ignore"),
            make_column_selector(dtype_include="str"),
        ),
        (
            "Age",
            KBinsDiscretizer(n_bins=5, encode="onehot", strategy="quantile"),
            ["age"],
        ),
        ("Balance", PowerTransformer(method="yeo-johnson"), ["balance"]),
    ],
    remainder="passthrough",
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_reg_pipe = Pipeline(
    [("preprocessor", ct), ("model", LogisticRegression(max_iter=1000))]
)

In [ ]:
proba = cross_val_predict(
    log_reg_pipe, X_train, y_train, cv=cv, method="predict_proba"
)[:, 1]
print(proba.shape)

In [ ]:
precision, recall, threshold = precision_recall_curve(y_train, proba)
ap = average_precision_score(y_train, proba)
plt.plot(recall, precision, label=f"PR-AUC (AP) = {ap:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve")
plt.legend()
plt.show()

To make this project more interesting i will introduce two variables which will impact decision making.
r = revenue from successful call = 100$ c
c = cost of the call = 10$
Also our call-center can only call 2000 customer a day because we have a little team of callers.

In [ ]:
r = 100
c = 10

thresholds = np.linspace(0.05, 0.95, 91)
results = []

for t in thresholds:
    pred = proba >= t
    tp = ((pred) & (y_train == 1)).sum()
    calls = pred.sum()
    prec = tp / calls if calls > 0 else 0
    rec = tp / (y_train == 1).sum()
    money = tp * r - calls * c
    results.append(
        {
            "threshold": t,
            "money": money,
            "n_calls": calls,
            "precision": prec,
            "recall": rec,
        }
    )

df = pd.DataFrame(results)
best_threshold_nolimit = df.loc[df["money"].idxmax()]
print(f"The best threshold for unlimited calls: {best_threshold_nolimit['threshold']}")
df2000 = df[df["n_calls"] <= 2000]
best_threshold_limit = df2000.loc[df2000["money"].idxmax()]
print(f"The best threshold for 2000 calls: {best_threshold_limit['threshold']}")

The best economic situation is on threshold of 0.11 but in this scenario we need to make 12k calls a day. Otherwise the most optimal threshold which our call-center can use is 0.36 with 2k calls.

In [ ]:
with open("../models/threshold.json", "w") as f:
    json.dump(best_threshold_nolimit.to_dict(), f, indent=2)